Ethylene ($C_2H_4$) is a monomer for producing various polymers. However, most $C_2H_4$ streams contain a small amount of acetylene ($C_2H_2$) which poisons the catalyst used for polymerization. We need to selectively hydrogenate $C_2H_2$ to $C_2H_4$, while minimizing complete hydrogenation to ethane ($C_2H_6$). This reaction is performed on $Pd/Al_2O_3$ in a CSTR operating at steady state. Assume differential conversion.

Reaction mechanism: 
1) $H_2 + 2[*] \rightleftharpoons 2H^*$ 
2) $C_2H_2 + [*] \rightleftharpoons C_2H_2^*$ 
3) $C_2H_2^* + H^* \rightleftharpoons C_2H_3^* + [*]$ 
4) $C_2H_3^* + H^* \rightleftharpoons C_2H_4^* + [*]$ 
5) $C_2H_4^* \rightleftharpoons C_2H_4 + [*]$ 
6) $C_2H_4^* + H^* \rightleftharpoons C_2H_5^* + [*]$ 
7) $C_2H_5^* + H^* \rightleftharpoons C_2H_6 + 2[*]$

In [1]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import least_squares
import pandas as pd

# Constants
R = 8.314        # [J/mol.K] 
kb_J = 1.38e-23  # [J/K]
kb_eV = 8.617e-5 # [eV/K]
h = 6.626e-34    # [J.s]
N_a = 6.022e23   # [1/mol]

# Reactor and inlet feed conditions 
T = 350 # Operating temperature [K]
P = 1 # Pressure [bar]
y_in = np.zeros(4)
y_in[0] = 0.1 # H2
y_in[1] = 0.01 # C2H2
y_in[2] = 0.7 # C2H4
y_in[3] = 0.02 # C2H6
n_gas = len(y_in)

## Thermodynamics and kinetics

In [2]:
# All G's are reported at standard state (1 bar, 298K, 1cm2 Pd). Assume G's are independent of T. 

# Reaction free energies (deltaG) in kJ/mol
deltaG = np.zeros(7)
deltaG[0] = -30 # H2 dissociative adsorption energy
deltaG[1] = -25 # C2H2 adsorption energy
deltaG[2] = -10 # First H addition
deltaG[3] = -10 # Second H addition
deltaG[4] = -5 # C2H4 desorption energy
deltaG[5] = 5 # Third H addition
deltaG[6] = -15 # Fourth H addition and C2H6 desorption 

# Activation free energies (Gact) in kJ/mol
Gact = np.zeros(7)
Gact[0] = 5 # H2 dissociative adsorption
Gact[1] = 5 # C2H2 adsorption
Gact[2] = 15 # First H addition
Gact[3] = 15 # Second H addition
Gact[4] = 20 # C2H4 desorption
Gact[5] = 40 # Third H addition
Gact[6] = 15 # Fourth H addition and C2H6 desorption

K_eq = np.exp(-deltaG*1e3/(R*T))           # Equilibrium constants 
kf = kb_J*T/h * np.exp(-Gact*1e3/(R*T))    # Forward rate constants
kb = kf/K_eq                               # Backward rate constants

In [3]:
# Stoichiometric matrix (rows are species, columns are reactions)
nu_matrix = np.array([[-1,  0,  0,  0,  0,  0,  0],  # H2
                      [ 0, -1,  0,  0,  0,  0,  0],  # C2H2
                      [ 0,  0,  0,  0,  1,  0,  0],  # C2H4
                      [ 0,  0,  0,  0,  0,  0,  1],  # C2H6
                      [ 2,  0, -1, -1,  0, -1, -1],  # H*
                      [ 0,  1, -1,  0,  0,  0,  0],  # C2H2*
                      [ 0,  0,  1, -1,  0,  0,  0],  # C2H3*
                      [ 0,  0,  0,  1, -1, -1,  0],  # C2H4* 
                      [ 0,  0,  0,  0,  0,  1, -1],  # C2H5*
                      [-2, -1,  1,  1,  1,  1,  2]]) # Empty sites

# Isolate reactant and product stoichiometric coefficients for use in rate expressions
react_matrix = np.where(nu_matrix < 0, -nu_matrix, 0) 
prod_matrix = np.where(nu_matrix > 0, nu_matrix, 0)    

# Initial guesses for the solver 
initial_guess = np.array([0.8, # H*
                          0.05, # C2H2*
                          0.05, # C2H3*
                          0.02, # C2H4*
                          0.03, # C2H5*
                          0.05]) # Empty sites

In [4]:
# Solver function setup with log transformation of coverages
def ss_cstr_mkm(x_log): 

    theta = np.exp(x_log) # Transform back to linear space 

    # Calculate activities
    P_gas = P * y_in
    activities = np.concatenate((P_gas, theta))
    
    # Calculate elementary reaction rates with reactant and product stoichiometric coefficient matrices
    r_f = kf * np.prod(activities[:, None] ** react_matrix, axis=0)
    r_b = kb * np.prod(activities[:, None] ** prod_matrix, axis=0)
    r_net = r_f - r_b 
    
    # Matrix multiplication to calculate rates for each species from elementary reaction rates 
    species_rates = nu_matrix @ r_net 

    # Differential conversion: residuals are only needed for surface species
    residuals = species_rates[n_gas:-1]
    scaling_factor = np.max(r_f) + 1e-20 # Scaling residuals to improve convergence 
    residuals = residuals / scaling_factor
    
    # Add empty site balance (last equation)
    residuals = np.append(residuals, (np.sum(theta) - 1.0))
    return residuals

initial_guess_log = np.log(initial_guess) # Convert initial guess to log space
solution_log = least_squares(ss_cstr_mkm, initial_guess_log, 
                             ftol=1e-12, xtol=1e-12, gtol=1e-12, 
                             x_scale='jac', bounds=(-np.inf, 0),
                             max_nfev = 3000) # Solve for log coverage, lot of changed options

class SolutionWrapper: pass # Optional wrapper (laziness)
solution = SolutionWrapper()
solution.x = np.exp(solution_log.x) # Converted final solution back to physical linear coverages

In [5]:
def calculate_rates(theta):

    # Calculate activities
    P_gas = P * y_in
    activities = np.concatenate((P_gas, theta))

    # Calculate forward and backward rates
    r_f = kf * np.prod(activities[:, None] ** react_matrix, axis=0)
    r_b = kb * np.prod(activities[:, None] ** prod_matrix, axis=0)
    return r_f, r_b

rate_forward, rate_backward = calculate_rates(solution.x)

## Solutions

In [6]:
# Display solutions 
theta = solution.x
surf_names = ['H*', 'C2H2*', 'C2H3*', 'C2H4*', 'C2H5*', 'Empty (*)']
species = surf_names
outlet = [f"{val:.8f}" for val in theta]

df_sol = pd.DataFrame({
    'Species': species, 
    'Coverage': outlet})
display(df_sol)

,Species,Coverage
0,H*,0.61269972
1,C2H2*,0.05070885
2,C2H3*,0.05141179
3,C2H4*,0.18476840
4,C2H5*,0.00003620
5,Empty (*),0.10037503


## Reversibility

In [7]:
reversibility = rate_backward / rate_forward

df_rev = pd.DataFrame({
    'Reaction Number': range(1, len(reversibility) + 1),
    'Reversibility': [f"{val:.4f}" for val in reversibility]})
display(df_rev)

,Reaction Number,Reversibility
0,1,0.0124
1,2,0.0094
2,3,0.0053
3,4,0.0189
4,5,0.0682
5,6,0.0002
6,7,0.0524


## DRC (C2H4)

In [8]:
# Original net rates for C2H4 production 
r_C2H4 = rate_forward[4] - rate_backward[4]

epsilon = 1e-4 # Perturbation factor for DRC calculation
DRC_C2H4 = np.zeros(7)

# Save original rate constants
kf_orig = np.copy(kf); kb_orig = np.copy(kb)

# Calculate DRCs
for i in range(7):

    # Increase kf and adjust kb while keeping K_eq constant
    kf[i] = kf_orig[i] * (1 + epsilon)
    kb[i] = kf[i] / K_eq[i]
    
    sol_pert_log = least_squares(ss_cstr_mkm, solution_log.x, 
                                 ftol=1e-12, xtol=1e-12, gtol=1e-12, 
                                 x_scale='jac', bounds=(-np.inf, 0),
                                 max_nfev = 3000) 
    rate_forward_pert, rate_backward_pert = calculate_rates(np.exp(sol_pert_log.x))
    
    # Calculate perturbed net rates
    r_C2H4_pert = rate_forward_pert[4] - rate_backward_pert[4]
    
    # Calculate DRC: (k/r)*(dr/dk) => (delta_r/r_ss)/epsilon
    DRC_C2H4[i] = (r_C2H4_pert - r_C2H4) / (r_C2H4 * epsilon)
        
    # Restore original constants
    kf[i] = kf_orig[i]; kb[i] = kb_orig[i]

# Print results
reaction_names = [
    "1: H2 + 2* <-> 2H*",
    "2: C2H2 + * <-> C2H2*",
    "3: C2H2* + H* <-> C2H3* + *",
    "4: C2H3* + H* <-> C2H4* + *",
    "5: C2H4* <-> C2H4 + *",
    "6: C2H4* + H* <-> C2H5* + *",
    "7: C2H5* + H* <-> C2H6 + 2*"]
df_drc = pd.DataFrame({
    'Reaction Step': reaction_names,
    'DRC (C2H4)': [f"{val:.4f}" for val in DRC_C2H4]})
display(df_drc)
print(f"Sum of DRCs: {np.sum(DRC_C2H4):.4f}")

,Reaction Step,DRC (C2H4)
0,1: H2 + 2* <-> 2H*,-0.9442
1,2: C2H2 + * <-> C2H2*,1.9039
2,3: C2H2* + H* <-> C2H3* + *,0.0221
3,4: C2H3* + H* <-> C2H4* + *,0.0043
4,5: C2H4* <-> C2H4 + *,0.0142
5,6: C2H4* + H* <-> C2H5* + *,-0.0000
6,7: C2H5* + H* <-> C2H6 + 2*,0.0000


Sum of DRCs: 1.0002


In [9]:
solution_log

     message: `gtol` termination condition is satisfied.
     success: True
      status: 1
         fun: [ 6.839e-16  3.420e-16 -1.710e-16 -3.420e-16  5.009e-19
               -1.110e-16]
           x: [-4.899e-01 -2.982e+00 -2.968e+00 -1.689e+00 -1.023e+01
               -2.299e+00]
        cost: 3.7161664488661387e-31
         jac: [[-1.936e+00 -9.374e-01 ... -6.692e-04  3.802e+00]
               [-9.374e-01 -9.463e-01 ... -0.000e+00  9.463e-01]
               ...
               [-3.498e-05 -0.000e+00 ... -6.694e-04  7.007e-05]
               [ 6.127e-01  5.071e-02 ...  3.620e-05  1.004e-01]]
        grad: [-2.035e-15 -1.131e-15 -8.123e-16  3.366e-16 -4.621e-19
                2.894e-15]
  optimality: 3.371253858353349e-15
 active_mask: [0 0 0 0 0 0]
        nfev: 10
        njev: 10